### LLM Gateway Explained — Build One With LiteLLM + LangChain

In this hands-on tutorial, we'll cover:

- What is an LLM Gateway? — The problem it solves
- Why do we need it? — Real production pain points
- Core capabilities — Routing, fallbacks, caching, observability, cost tracking
- Practical implementation — Build one from scratch using LiteLLM
- Integration with LangChain — Plug the gateway into your agentic apps
- Production patterns — Logging, retries, multi-provider fallbacks

### Part 1: What is an LLM Gateway?

LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

1. Without a Gateway :
- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query
2. With a Gateway :
- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

### Part 2: Installation & Setup
We'll use:

- LiteLLM → the most popular open-source LLM gateway (supports 100+ providers)
- LangChain → for building agentic workflows on top of the gateway
- python-dotenv → for managing API keys

In [3]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [4]:
import litellm
litellm.suppress_debug_info = True

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "OK" if os.getenv("OPENAI_API_KEY") else "Not Available")
print("Anthropic key loaded: ", "OK" if os.getenv("ANTHROPIC_API_KEY") else "Not available")
print("Groq key loaded:      ", "OK" if os.getenv("GROQ_API_KEY") else "Not available")
print("Google key loaded:      ", "OK" if os.getenv("GOOGLE_API_KEY") else "Not available")
print("Mistral key loaded:      ", "OK" if os.getenv("MISTRAL_API_KEY") else "Not available")

OpenAI key loaded:     OK
Anthropic key loaded:  Not available
Groq key loaded:       OK
Google key loaded:       OK
Mistral key loaded:       OK


### Part 3: The Simplest LiteLLM Example — Unified API

The biggest pain point: every provider has a different SDK.

- LiteLLM gives you one function — completion() — that works with all of them.

In [15]:
from litellm import completion

# Same code, different providers — just change the `model` string!
response_google = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("GoogleAI: ", response_google.choices[0].message.content)
# Call Groq (super fast inference)
response_groq = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("GroqAI: ", response_groq.choices[0].message.content)

GoogleAI:  Retrieval-Augmented Generation (RAG) is a technique that improves the accuracy and relevance of AI models by retrieving authoritative, external data to supplement the model’s internal knowledge before it generates a response.
GroqAI:  RAG (Retrieval‑Augmented Generation) is a technique that blends a document‑retrieval system with a language model, letting the model generate responses that are grounded in up‑to‑date, retrieved text.


In [17]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("OpenAI",     "gpt-4o-mini"),
    ("Groq",       "groq/openai/gpt-oss-20b"),
    ("Anthropic",  "claude-3-5-haiku-20241022"),
    ("Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: {type(e).__name__}")

OpenAI         : RateLimitError
Groq           : Retrieval‑Augmented Generation (RAG) is a neural approach that first fetches rel
Anthropic      : BadRequestError
Gemini         : NotFoundError


### Part 4: Automatic Fallbacks — When OpenAI Goes Down

Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.


In [18]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "groq/openai/gpt-oss-20b",
        "gemini/gemini-3.1-flash-lite"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

15:34:26 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "a:\langchainupdated\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2836, in async_completion
    response: Final = await client.post(
                      ^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "a:\langchainupdated\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 340, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "a:\langchainupdated\.venv\Lib\site-package

Response: ## What Is an LLM Gateway?

An **LLM Gateway** (Large Language Model Gateway) is a middleware layer that sits between an application (or a set of applications) and one or more LLMs (e.g., GPT‑4, Claud ...

Which model actually answered? openai/gpt-oss-20b


If "openai/gpt-oss-20b" is rate-limited or down, LiteLLM transparently retries with Claude, then Groq. Your app never sees the failure.

- This is the #1 reason teams adopt an LLM Gateway.

In [ ]:
from litellm import completion

# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # will fail intentionally
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",                              # 1st backup: real OpenAI model
        "groq/llama-3.3-70b-versatile"              # 2nd backup: Groq
    ]
)

print("App still got a response, even though the primary failed!")
print(f"\nModel that actually answered: {response.model}")
print(f"\nResponse: {response.choices[0].message.content[:200]}...")

### Part 5: Cost Tracking — Know Where Your Money Goes

LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [19]:
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Silicon wakes to life,
Thinking thoughts of code and light,
Mind without a soul.

Input tokens:  8
Output tokens: 19
Cost:         $0.00003050


Now imagine running this across thousands of calls per day, tagged by team or project — you instantly know who's burning the budget.


### Part 6: Caching — Don't Pay Twice for the Same Question

- If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.

- Enable in-memory caching with one line:

In [23]:
import litellm

# Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("LiteLLM state reset — ready for clean caching demo")

LiteLLM state reset — ready for clean caching demo


In [26]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="gemini/gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\nSpeedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")


First call (API):   1.38s — LLM stands for Large Language Model.
Second call (cache): 0.2125s — LLM stands for Large Language Model.

Speedup: 6.5x faster, and ZERO cost on the second call!


### Part 7: Smart Routing — The Right Model for the Right Job

Why use one model for everything?

- Coding tasks → Claude Sonnet
- Cheap summaries → GPT-4o-mini
- Super fast replies → Groq Llama
- Complex reasoning → Claude Opus

Use LiteLLM's Router to define routing rules:

In [ ]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",                                      
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gemini/gemini-3.1-flash-lite",
            "api_key": os.getenv("GOOGLE_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("Fast/cheap (Groq-20b): ", fast_response.choices[0].message.content[:150])
print("\nSmart/coding (Groq-120b):\n", code_response.choices[0].message.content[:300])

Fast/cheap (Groq):  **AI’s Impact on Software Development – A Quick Summary**

1. **Accelerated Coding**
   - *Code generation*: Large language models (e.g., GitHub Copil

Smart/coding (GPT-4o):
 Here’s a tiny, fully‑typed Python function that takes a string and returns a new string that is the reverse of the input.  
It shows two common ways: slicing (Pythonic) and a manual loop (illustrative).

```python
# Python 3.11+

from typing import Final

def reverse_string(s: str) -> str:
    """
 


### Part 8: Load Balancing Across Multiple API Keys

Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [28]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gemini/gemini-3.1-flash-lite",
            "api_key": os.getenv("GOOGLE_API_KEY"),
        },
        "model_info": {"id": "gemini-3.1flash-lite"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-openai-20b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(5):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        gemini-3.1flash-lite    1108 ms   Hello! I am ready to help. What is 
#2        gemini-3.1flash-lite    1032 ms   Hello! I am ready to help. What is 
#3        gemini-3.1flash-lite     930 ms   Hello! I am ready to help. Please l
#4        groq-openai-20b         2101 ms   Hello! What kind of list would you 
#5        groq-openai-20b          945 ms   Hello! Could you please tell me the
